# Exercises

Run notebooks 00–06 first so `data/kg/mentions.jsonl` and `data/kg/eval.json` exist. These cells use the saved mentions. They do not retag the book.

## 1. Three held-out misses

List three proper-name surfaces in chapters 14–27 that match exactly one gold entity and whose predicted id is different. Print the chapter, the surface, the predicted id, the gold id, and the link method.

Read the link method on each row. `exact` means the surface shared its tokens with a cluster whose id is not the gold id. A bare `John` can win that cluster's label while `Dr. John Seward` sits inside it. `None` would mean the structural rules left the mention alone.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "kg").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(ROOT)


In [ ]:
import json

from src.kg.config import load_yaml
from src.kg.evaluate import match_gold
from src.kg.mentions import Mention

gold = load_yaml(ROOT / "data" / "gold" / "entities.yml")["entities"]
mentions = [
    Mention(**json.loads(line))
    for line in (ROOT / "data" / "kg" / "mentions.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]

misses = []
seen = set()
for mention in mentions:
    if mention.mention_class != "PROP" or mention.chapter_number is None or mention.chapter_number < 14:
        continue
    hits = match_gold(mention, gold)
    if len(hits) != 1 or mention.entity_id == hits[0]:
        continue
    key = (mention.surface, mention.entity_id, hits[0])
    if key in seen:
        continue
    seen.add(key)
    misses.append((mention.chapter_number, mention.surface, mention.entity_id, hits[0], mention.link_method))

print(len(misses), "distinct missed surfaces in chapters 14-27")
for row in misses[:3]:
    print(row)


## 2. A false appositive must stay split

`Lucy Westenra, Mrs. Harker arrived.` is two people who happen to be named in a row. Comma apposition may merge a name with a rank title (`Lord Godalming`). It must refuse `Mrs`, `Mr`, or `Miss` when the surnames differ.

The cell prints `split` or `MERGED — file this as a bug`.


In [ ]:
from src.kg.entries import Entry
from src.kg.mentions import Mention
from src.kg.resolve import resolve_mentions

text = "Lucy Westenra, Mrs. Harker arrived."
cfg = {
    "auto_merge_threshold": 0.85,
    "review_threshold": 0.60,
    "jaro_block": 0.92,
    "rare_token_max_surfaces": 30,
    "weights": {"jaro": 0.4, "surname": 0.3, "given": 0.2, "honorific": 0.1},
    "male_givens": ["jonathan", "arthur"],
    "female_givens": ["mina", "lucy"],
}
spans = [("Lucy Westenra", 0, 13), ("Mrs. Harker", 15, 26)]
assert text[0:13] == "Lucy Westenra"
assert text[15:26] == "Mrs. Harker"
mentions = [
    Mention(f"m-{start}", 1, 1, 0, start, end, surface, "PER", "PROP", "notebook")
    for surface, start, end in spans
]
entry = Entry(1, 1, "Journal", "Mina Harker", None, None, "journal", text, 1)
resolved = resolve_mentions(mentions, [entry], cfg)
same = resolved.mentions[0].entity_id == resolved.mentions[1].entity_id
print("MERGED — file this as a bug" if same else "split")


## 3. Recap

Blocking keeps pair scoring inside candidates that already share a rare token or a high Jaro-Winkler. Average linkage then merges a block only at 0.85 or above, and only with surname or rare-token support. Cannot-links, including two full names with different given names, stop the merge.

Pronouns attach inside one sentence when that sentence has a single compatible person. They do not cross chapters. Most pronouns stay unresolved. That is visible in the link-method counts below.

`castle_dracula` and `count_dracula` are different ids. Mina's narrator labels stay as two strings on the entries and one entity in the graph. Kinship patterns require the keyword in the gap between two adjacent mentions. This novel does not put the keyword there, so the pattern list is empty. The relationships in the graph come from `data/gold/relations.yml`, each with a quote the build finds in the text, plus the curated `part_of` edge.


In [ ]:
import json
from collections import Counter

from src.kg.build import documents
from src.kg.config import load_config

mentions = [
    Mention(**json.loads(line))
    for line in (ROOT / "data" / "kg" / "mentions.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
graph = json.loads((ROOT / "data" / "kg" / "graph.json").read_text(encoding="utf-8"))
entries = documents(ROOT, load_config(ROOT / "configs" / "kg.yml"))

pronouns = [m for m in mentions if m.mention_class == "PRON"]
print("pronoun link methods", Counter(m.link_method for m in pronouns))
print("narrator labels", sorted({e.narrator for e in entries if e.narrator and e.narrator.startswith("Mina")}))
by_id = {node["id"]: node for node in graph["nodes"]}
print("castle type", by_id["castle_dracula"]["entity_type"], "count type", by_id["count_dracula"]["entity_type"])
print("ids differ", by_id["castle_dracula"]["id"] != by_id["count_dracula"]["id"])
for edge in graph["edges"]:
    if edge["type"] == "RELATION":
        print(edge["rel"], edge["source"], "->", edge["target"], edge.get("method", "curated"))
